In [ ]:
# 셀 1에서 설치 먼저
!pip install selenium webdriver-manager

In [ ]:
# 새 셀에서 실행
import os
print(os.getcwd())  # 현재 작업 디렉토리 확인

In [ ]:
# 셀 1: 라이브러리 설치
!pip install requests beautifulsoup4 pandas

In [ ]:
# 셀 2: 임포트 & 설정
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random

INPUT_CSV = 'cleaned_reviews.csv'  # 파일 경로 맞게 수정
OUTPUT_CSV = 'ingredients.csv'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'ko-KR,ko;q=0.9',
    'Referer': 'https://www.oliveyoung.co.kr/',
}

print('설정 완료!')

In [ ]:
# 셀 3: CSV에서 고유 제품 + goodsNo 추출
df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')

# goodsNo 추출
df['goodsNo'] = df['리뷰URL'].str.extract(r'goodsNo=([A-Z0-9]+)')

# 제품별 중복 제거
products_df = df[['제품명', '브랜드', '카테고리', 'goodsNo']].drop_duplicates(subset='goodsNo').reset_index(drop=True)

print(f'고유 제품 수: {len(products_df)}개')
products_df.head()

In [ ]:
# 셀 4 전체를 이걸로 교체

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager
import time, random

options = webdriver.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def crawl_ingredients(goods_no):
    url = f'https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo={goods_no}'
    try:
        driver.get(url)
        time.sleep(2)

        # 페이지 맨 아래로 스크롤 (버튼 활성화 유도)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)

        # '상품정보 제공고시' 텍스트 포함된 버튼 찾아서 JS 클릭
        buttons = driver.find_elements(By.XPATH, '//*[contains(text(), "상품정보 제공고시")]')
        if buttons:
            driver.execute_script("arguments[0].click();", buttons[0])
            time.sleep(1.5)

        # XPath로 성분 추출
        el = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="tab-panels"]/section/ul/li[1]/div/div/table/tbody/tr[7]/td'))
        )
        return el.text.strip()

    except Exception as e:
        return f'오류:{str(e)}'

print('드라이버 준비 완료!')

In [ ]:
# 셀 5 테스트
test_row = products_df.iloc[0]
print(f'제품명: {test_row["제품명"]}')
result = crawl_ingredients(test_row['goodsNo'])
print(f'전성분: {result[:300]}')

In [ ]:
# 셀 6 - 전체 크롤링

ingredients_list = []

for i, row in products_df.iterrows():
    result = crawl_ingredients(row['goodsNo'])
    ingredients_list.append(result)

    # 매 10개마다 진행상황 출력
    if (i + 1) % 10 == 0:
        print(f'[{i+1}/{len(products_df)}] {row["제품명"][:30]} → {result[:50]}')

    # 50개마다 중간 저장
    if (i + 1) % 50 == 0:
        temp_df = products_df.iloc[:i+1].copy()
        temp_df['전성분'] = ingredients_list
        temp_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print(f'  💾 중간 저장 완료 ({i+1}개)')

    time.sleep(random.uniform(2.0, 3.5))

# 최종 저장
products_df['전성분'] = ingredients_list
products_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n✅ 완료! {OUTPUT_CSV} 저장됨')

In [ ]:
# 셀 7: 결과 확인
result_df = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')

total = len(result_df)
no_data = result_df['전성분'].str.contains('없음|오류|타임아웃').sum()

print(f'전체 제품 수: {total}개')
print(f'성분 수집 성공: {total - no_data}개')
print(f'성분 정보 없음: {no_data}개')
print()
result_df.head()